### Data Engineering: Data Ingestion for Logs

In [ ]:
import os
import boto3
from datetime import datetime, timedelta

# ---------- CONFIG ----------
LOGS_FOLDER = "day-wise-logs-data/"
DATE_TRACKER_FILE = "log_date_tracker.txt"

S3_BUCKET = "care-ticket-data" 
S3_PREFIX = "support-logs/raw/"

from dotenv import load_dotenv
load_dotenv()


'eu-north-1'

In [5]:


AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [6]:
# ---------- UTILITY FUNCTIONS ----------

def read_last_date(file_path):
    """
    Read the last successfully processed log date from the log date tracker file.

    Args:
        file_path (str): Path to the local date tracker file.

    Returns:
        str: Last processed date in YYYY-MM-DD format.
    """
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            return f.read().strip()

    # Default date used when the pipeline runs for the first time.
    return "2025-06-30"


def update_last_date(file_path, new_date):
    """
    Update the tracker file with the most recently processed date.

    This allows the pipeline to remember its progress and continue
    incrementally on the next run.

    Args:
        file_path (str): Path to the local date tracker file.
        new_date (str): Successfully processed date in YYYY-MM-DD format.

    Returns:
        None
    """
    with open(file_path, "w") as f:
        f.write(new_date)


def get_next_date(last_date_str):
    """
    Calculate the next date to process in the ingestion pipeline.

    Args:
        last_date_str (str): Last processed date in YYYY-MM-DD format.

    Returns:
        str: Next date to process in YYYY-MM-DD format.
    """
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)

    return next_date.strftime("%Y-%m-%d")


def upload_log_file_to_s3(file_path, bucket, key):
    """
    Upload a local support log file to Amazon S3.

    The function reads the raw log file as text and uploads its contents
    to the specified S3 bucket and object key.

    Args:
        file_path (str): Path to the local log file.
        bucket (str): Destination S3 bucket name.
        key (str): Destination object key/path within the S3 bucket.

    Returns:
        None
    """
    # Create an S3 client using credentials loaded from environment variables.
    s3 = boto3.client("s3", **AWS_CONFIG)

    # Read the raw log file without modifying its contents.
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Upload the log content to the S3 raw data layer.
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=content
    )

    print(f"✅ Uploaded log file to s3://{bucket}/{key}")


# ---------- MAIN INGESTION LOGIC ----------

def run_log_ingestion():
    """
    Run one incremental daily log-ingestion cycle.

    The function:
        1. Reads the last successfully processed date.
        2. Calculates the next date to process.
        3. Builds the expected daily log filename.
        4. Builds the destination S3 object key.
        5. Uploads the log file to Amazon S3.
        6. Updates the tracker after a successful upload.

    Returns:
        None
    """
    # Determine which day's log file should be processed next.
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    print(f"Last processed date: {last_date}")
    print(f"Next processing date: {next_date}")

    # Build the expected source log filename for the next processing date.
    file_name = f"support_logs_{next_date}.log"
    log_file_full_path = os.path.join(LOGS_FOLDER, file_name)

    print(f"Source log file: {log_file_full_path}")

    # Build the S3 destination path while preserving the daily filename.
    s3_key = f"{S3_PREFIX}support_logs_{next_date}.log"

    # Upload the daily log file to the S3 raw layer.
    upload_log_file_to_s3(
        log_file_full_path,
        S3_BUCKET,
        s3_key
    )

    # Advance the tracker only after the upload completes successfully.
    update_last_date(DATE_TRACKER_FILE, next_date)

    print(f"📅 Updated tracker to {next_date}")

In [38]:
# ------- RUN -------
if __name__ == "__main__":
    run_log_ingestion()

Last processed date: 2025-07-31
Next processing date: 2025-08-01
Source log file: day-wise-logs-data/support_logs_2025-08-01.log


FileNotFoundError: [Errno 2] No such file or directory: 'day-wise-logs-data/support_logs_2025-08-01.log'